# Player bounding

Load 20 frames from a SportsMOT training sequence for player-detection work.

In [ ]:
from io import BytesIO
from pathlib import Path
import tarfile

from PIL import Image

archive_candidates = [
    Path("data/sportsmot_download/dataset/train.tar"),
    Path("../data/sportsmot_download/dataset/train.tar"),
]
archive = next((path for path in archive_candidates if path.exists()), None)

if archive is None:
    raise FileNotFoundError("Could not find data/sportsmot_download/dataset/train.tar")

sequence = "train/v_-6Os86HzwCs_c001"

with tarfile.open(archive, "r") as tar:
    selected = sorted(
        (member for member in tar.getmembers()
         if member.isfile()
         and member.name.startswith(sequence + "/img1/")
         and member.name.endswith(".jpg")
         and not member.name.rsplit("/", 1)[-1].startswith("._")),
        key=lambda member: member.name,
    )[:20]

    images = [
        Image.open(BytesIO(tar.extractfile(member).read())).convert("RGB")
        for member in selected
    ]

print(f"Loaded {len(images)} frames from {sequence}")
print(f"First frame: {selected[0].name}")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 7))
plt.imshow(images[0])
plt.axis("off")
plt.show()

### Load Ground Truth
The GT file uses MOT format, one row per player per frame: `frame, track_id, x, y, width, height, conf, class, visibility`. Frames are 1-indexed and `(x, y)` is the top-left corner.

In [ ]:
import io
import tarfile
import numpy as np

sequence = "train/v_-6Os86HzwCs_c001"

with tarfile.open(archive, "r") as tar:
    gt_bytes = tar.extractfile(f"{sequence}/gt/gt.txt").read()

gt = np.loadtxt(io.BytesIO(gt_bytes), delimiter=",")

### Load Models
Loading multiple ultralytics models to compare results. YOLO11 (nano, small, medium, large, xlarge) and RT-DETR-x are pretrained on COCO, where class 0 is person.

In [ ]:
%pip install ultralytics

In [ ]:
from pathlib import Path
from ultralytics import YOLO, RTDETR

# Create the subfolder for model weights
weights_dir = Path("player_bounding_models")
weights_dir.mkdir(exist_ok=True)

models = {
  "yolo11_nano": YOLO(weights_dir / "yolo11n.pt"),
  "yolo11_small": YOLO(weights_dir / "yolo11s.pt"),
  "yolo11_med": YOLO(weights_dir / "yolo11m.pt"),
  "yolo11_large": YOLO(weights_dir / "yolo11l.pt"),
  "yolo11_xlarge": YOLO(weights_dir / "yolo11x.pt"),
  "rtdetr": RTDETR(weights_dir / "rtdetr-x.pt")
}

### First Frame Check 
Run model predictions on first frame and draw bounding boxes for predictions and gt to visually compare. Red boxes are ground truth. Green boxes are model predictions. Only class 0 (person) predictions are drawn.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2

frame_gt = gt[gt[:, 0] == 1]
first_frame_results = {}
for model_name, model in models.items():
  first_frame_results[model_name] = model.predict(images[0], conf=0.5)[0]

# red = ground truth, green = predictions
for model_name, model_results in first_frame_results.items():
    img = np.array(images[0])
    cv2.putText(img, f"Model: {model_name}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    for row in frame_gt:
        _, id, x, y, w, h = row[:6]
        x1, y1, x2, y2 = int(x), int(y), int(x + w), int(y + h)
        cv2.rectangle(img, (x1, y1), (x2, y2), (255, 0, 0), 2)
        cv2.putText(img, f"ID {int(id)}", (x1, y2 + 15), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)
    
    for i, box in enumerate(model_results.boxes):
        if int(box.cls[0]) != 0: # skip if class isn't person
            continue

        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
        cv2.putText(img, f"ID {i}", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
        
    plt.imshow(img)
    plt.axis("off")
    plt.show()

### Matching Predictions to Ground Truth

- **IoU (Intersection over Union)** measures how much two boxes overlap: shared area divided by combined area, from 0 to 1.
- A prediction is a true positive (TP) if it matches a ground-truth box with IoU ≥ 0.5.
- Matching is one-to-one meaning each box can be matched only once, and the highest-overlap pairs are matched first.
- False positives (FP) are predictions that don't match ground truth. False negatives (FN) are unmatched ground-truth boxes.
- **Precision** = TP / (TP + FP): of the boxes the model drew, how many were real players.
- **Recall** = TP / (TP + FN): of the real players, how many the model found.

In [ ]:
def iou(box_a, box_b):
    """Intersection over union for two (x1, y1, x2, y2) boxes."""
    inter_x1, inter_y1 = max(box_a[0], box_b[0]), max(box_a[1], box_b[1])
    inter_x2, inter_y2 = min(box_a[2], box_b[2]), min(box_a[3], box_b[3])
    intersection = max(0, inter_x2 - inter_x1) * max(0, inter_y2 - inter_y1)
    area_a = (box_a[2] - box_a[0]) * (box_a[3] - box_a[1])
    area_b = (box_b[2] - box_b[0]) * (box_b[3] - box_b[1])
    return intersection / (area_a + area_b - intersection)

def match_frame(pred_boxes, gt_boxes, iou_threshold=0.5):
    """One-to-one matching of predictions to ground truth for one frame."""
    candidate_pairs = []
    for pred_idx, pred_box in enumerate(pred_boxes):
        for gt_idx, gt_box in enumerate(gt_boxes):
            candidate_pairs.append((iou(pred_box, gt_box), pred_idx, gt_idx))
    candidate_pairs.sort(reverse=True)  # highest IoU first

    matched_preds, matched_gts, matches = set(), set(), []
    for score, pred_idx, gt_idx in candidate_pairs:
        if score < iou_threshold:
            break
        if pred_idx in matched_preds or gt_idx in matched_gts:
            continue
        matched_preds.add(pred_idx)
        matched_gts.add(gt_idx)
        matches.append((pred_idx, gt_idx, score))

    true_positives = len(matches)
    false_positives = len(pred_boxes) - true_positives
    false_negatives = len(gt_boxes) - true_positives
    return true_positives, false_positives, false_negatives, matches

### Evaluate All Models on the Sequence

Frames are run one at a time so the GPU doesn't run out of memory. Predictions below `conf=0.5` are dropped before matching.

In [ ]:
def run_model_on_frames(model, frames):
    """Run a model on each frame separately to keep memory usage low."""
    return [model.predict(frame, conf=0.5, verbose=False)[0] for frame in frames]

# Per-frame lists of (x1, y1, x2, y2) bounding boxes predicted for each model
predicted_boxes_by_model = {
    model_name: [
        [tuple(box.xyxy[0].tolist()) for box in frame_result.boxes if int(box.cls[0]) == 0]
        for frame_result in run_model_on_frames(model, images)
    ]
    for model_name, model in models.items()
}

# Per-frame lists of ground-truth boxes
ground_truth_boxes_by_frame = [
    [(row[2], row[3], row[2] + row[4], row[3] + row[5]) for row in gt[gt[:, 0] == frame_number]]
    for frame_number in range(1, len(images) + 1)
]

for model_name, predicted_boxes_per_frame in predicted_boxes_by_model.items():
    total_tp = total_fp = total_fn = 0
    for pred_boxes, gt_boxes in zip(predicted_boxes_per_frame, ground_truth_boxes_by_frame):
        tp, fp, fn, _ = match_frame(pred_boxes, gt_boxes)
        total_tp += tp
        total_fp += fp
        total_fn += fn

    precision = total_tp / (total_tp + total_fp) if total_tp + total_fp else 0
    recall = total_tp / (total_tp + total_fn) if total_tp + total_fn else 0
    print(f"{model_name}: TP={total_tp} FP={total_fp} FN={total_fn} "
          f"precision={precision:.3f} recall={recall:.3f}")

## Results

Here are the results when I ran on the full sequence of 825 frames, conf = 0.5, IoU ≥ 0.5.

Frames evaluated: **825**. Ground-truth boxes: **8,074**.

| Model | TP | FP | FN | Precision | Recall |
|---|---|---|---|---|---|
| YOLO11 nano | 5137 | 2050 | 2937 | 0.715 | 0.636 |
| YOLO11 small | 6415 | 2389 | 1659 | 0.729 | 0.795 |
| YOLO11 medium | 6162 | 1862 | 1912 | 0.768 | 0.763 |
| YOLO11 large | 6687 | 2158 | 1387 | 0.756 | 0.828 |
| YOLO11 xlarge | 6540 | 1771 | 1534 | 0.787 | 0.810 |
| RT-DETR-x | 7185 | 3029 | 889 | 0.703 | 0.890 |

Nano is clearly the weakest and misses about twice as many players as the others.

xlarge has the best precision and the most balanced YOLO results.

RT-DETR has the best recall but the most false positives. Many are likely people outside the playing area, such as referees, coaches, or spectators. Filtering boxes by whether the bottom-center point (the feet) falls inside a court polygon could cut false positives and raise precision with little effect on recall, since real players on the court would still pass.

## Court Detection Filter

Many false positives (especially RT-DETR's) are likely referees, coaches, or spectators standing off the court. We can filter these out by estimating the court's playing area and keeping only boxes whose bottom-center point (the feet) falls inside it.

Approach: HSV color segmentation.
- Sample the floor color from a region near the bottom-center of the frame (assumed to be on the court).
- Threshold the whole frame around that color to get a floor mask.
- Take the largest contour's convex hull as the court polygon.
- Keep a prediction only if `cv2.pointPolygonTest` says its foot point is inside (or on) that polygon.

This is done per-frame so it tolerates camera pans/zooms, but it's sensitive to the sample region actually landing on the court and to lighting/arena color — worth a visual sanity check before trusting the numbers.

In [ ]:
import cv2
import numpy as np


def detect_court_polygon(
    frame,
    seed_points=tuple(
        (fx, fy)
        for fy in (0.45, 0.6, 0.75, 0.9, 0.97)
        for fx in (0.1, 0.25, 0.4, 0.5, 0.6, 0.75, 0.9)
    ),
    lo_diff=(12, 60, 60),
    up_diff=(12, 60, 60),
    min_area_frac=0.05,
    max_seed_area_frac=0.5,
):
    """Estimate the court polygon via seeded flood-fill on floor color.

    A single fixed sample point/region can land on a player, sideline, or logo and throw
    off a global color threshold, and a global threshold has no spatial constraint, so
    similarly-colored crowd/seats can merge into the "court" blob. Flood-filling from a
    seed point enforces connectivity to a known on-court location instead, with
    `cv2.FLOODFILL_FIXED_RANGE` so each pixel is compared against the fixed seed color
    rather than its already-filled neighbor -- otherwise small gradual shifts (floor ->
    shadow -> sideline -> crowd) accumulate and the fill "creeps" off the court entirely.

    A sparse set of seeds is in turn too fragile: a cluster of players standing shoulder to
    shoulder can split the visible floor into disconnected patches, and if no seed happens
    to land in one of those patches it's missed entirely (observed directly: real player
    foot points on one side of such a cluster were excluded). So a dense grid of seeds is
    flood-filled independently and unioned together -- safe to union since each individual
    fill is already connectivity-constrained to itself, and with enough seeds some will
    land in every disconnected floor patch. Any seed whose fill exceeds `max_seed_area_frac`
    of the frame is dropped first, in case that seed leaked into the crowd.

    Rather than requiring the union to form one connected blob (which the same
    player-cluster occlusion would still break), the polygon is the convex hull of every
    floor-colored pixel across the whole union, simplified via `cv2.approxPolyDP` for a
    cleaner, more stable boundary. Returns (polygon, mask); polygon is None if the union
    covers less than `min_area_frac` of the frame.
    """
    img = np.array(frame)
    hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
    h, w = hsv.shape[:2]

    union_mask = np.zeros((h, w), np.uint8)
    for fx, fy in seed_points:
        seed = (int(fx * w), int(fy * h))
        flood_fill_mask = np.zeros((h + 2, w + 2), np.uint8)
        flags = 8 | cv2.FLOODFILL_MASK_ONLY | cv2.FLOODFILL_FIXED_RANGE | (255 << 8)
        cv2.floodFill(hsv.copy(), flood_fill_mask, seed, newVal=0,
                       loDiff=lo_diff, upDiff=up_diff, flags=flags)
        mask = flood_fill_mask[1:-1, 1:-1]
        if np.count_nonzero(mask) <= max_seed_area_frac * h * w:
            union_mask = cv2.bitwise_or(union_mask, mask)

    if np.count_nonzero(union_mask) < min_area_frac * h * w:
        return None, union_mask

    mask = cv2.morphologyEx(union_mask, cv2.MORPH_CLOSE, np.ones((15, 15), np.uint8))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((7, 7), np.uint8))

    ys, xs = np.nonzero(mask)
    points = np.column_stack([xs, ys]).astype(np.int32)
    hull = cv2.convexHull(points)
    perimeter = cv2.arcLength(hull, True)
    polygon = cv2.approxPolyDP(hull, epsilon=0.015 * perimeter, closed=True)
    return polygon, mask


def box_in_court(box, court_polygon):
    """True if the box's bottom-center (feet) point falls inside/on the court polygon.

    Fails open (keeps the box) when no court polygon was detected, so a bad segmentation
    only removes the filter's benefit rather than discarding every prediction.
    """
    if court_polygon is None:
        return True
    x1, y1, x2, y2 = box
    foot_point = (float((x1 + x2) / 2), float(y2))
    return cv2.pointPolygonTest(court_polygon, foot_point, False) >= 0

### Visual Check
Visualize the detected floor mask and court polygon on the first frame.

In [ ]:
court_hull, court_mask = detect_court_polygon(images[0])

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].imshow(court_mask, cmap="gray")
axes[0].set_title("Floor mask")
axes[0].axis("off")

overlay = np.array(images[0]).copy()
if court_hull is not None:
    cv2.polylines(overlay, [court_hull], isClosed=True, color=(255, 0, 255), thickness=3)
axes[1].imshow(overlay)
axes[1].set_title("Court polygon (hull)")
axes[1].axis("off")

plt.show()

### Apply the Filter and Compare

Compute one court polygon per frame, drop any model prediction whose foot point falls outside it, then re-run the same precision/recall evaluation as before to compare filtered vs. unfiltered results.

In [ ]:
def evaluate(predicted_boxes_per_frame, ground_truth_boxes_by_frame):
    """Precision/recall for one model's predictions across all frames."""
    total_tp = total_fp = total_fn = 0
    for pred_boxes, gt_boxes in zip(predicted_boxes_per_frame, ground_truth_boxes_by_frame):
        tp, fp, fn, _ = match_frame(pred_boxes, gt_boxes)
        total_tp += tp
        total_fp += fp
        total_fn += fn

    precision = total_tp / (total_tp + total_fp) if total_tp + total_fp else 0
    recall = total_tp / (total_tp + total_fn) if total_tp + total_fn else 0
    return precision, recall, total_tp, total_fp, total_fn


court_polygons_by_frame = [detect_court_polygon(frame)[0] for frame in images]

filtered_predicted_boxes_by_model = {
    model_name: [
        [box for box in frame_boxes if box_in_court(box, court_polygon)]
        for frame_boxes, court_polygon in zip(predicted_boxes_per_frame, court_polygons_by_frame)
    ]
    for model_name, predicted_boxes_per_frame in predicted_boxes_by_model.items()
}

print(f"{'Model':<15}{'Precision':>12}{'Recall':>10}   ->  {'Filt. Precision':>16}{'Filt. Recall':>14}")
for model_name in models:
    base_p, base_r, *_ = evaluate(predicted_boxes_by_model[model_name], ground_truth_boxes_by_frame)
    filt_p, filt_r, *_ = evaluate(filtered_predicted_boxes_by_model[model_name], ground_truth_boxes_by_frame)
    print(f"{model_name:<15}{base_p:>12.3f}{base_r:>10.3f}   ->  {filt_p:>16.3f}{filt_r:>14.3f}")

## Court Filter Results

Ran on the same full sequence of 825 frames as above.

| Model | Precision | Recall | Filt. Precision | Filt. Recall |
|---|---|---|---|---|
| YOLO11 nano | 0.715 | 0.636 | 0.800 | 0.631 |
| YOLO11 small | 0.729 | 0.795 | 0.811 | 0.786 |
| YOLO11 medium | 0.768 | 0.763 | 0.852 | 0.758 |
| YOLO11 large | 0.756 | 0.828 | 0.858 | 0.819 |
| YOLO11 xlarge | 0.787 | 0.810 | 0.881 | 0.801 |
| RT-DETR-x | 0.703 | 0.890 | 0.839 | 0.873 |

The filter raises precision for every model by 8-14 points, at a cost of only 0.5-1.7 points of recall. Most false positives were off-court detections so cutting them barely touches recall. Recall drops slightly when a real player's foot point lands just outside the detected court polygon.

RT-DETR benefits most, it had the best recall but the worst precision unfiltered, and the filter brings its precision roughly in line with the YOLO models while keeping its recall lead.